In [ ]:
from sodapy import Socrata
import pandas as pd
from itertools import islice
import os
from pathlib import Path
import requests

In [4]:
# If the CTA data exists, read it in
if Path("output/cta_ridership.parquet").is_file():
    cta_df = pd.read_parquet("output/cta_ridership.parquet")
else:
    # If the CTA data does not exist, download it from Socrata
    client = Socrata(
        "data.cityofchicago.org",
        timeout=1000,
        app_token=None
    )

    cta_data = client.get_all('5neh-572f')

    chunk_size = 10_000
    chunks = []

    while True:
        chunk = list(islice(cta_data, chunk_size))
        print('Grabbing chunk of data...')
        if not chunk:
            break
        chunks.append(pd.DataFrame(chunk))

    cta_df = pd.concat(chunks, ignore_index=True)

    # And save as a flat file
    os.makedirs("output", exist_ok=True)
    cta_df.to_parquet(path='output/cta_ridership.parquet', engine='fastparquet', index=False)

In [ ]:
cta_df.head(10)

station_id     object
stationname    object
date           object
daytype        object
rides          object
dtype: object

In [ ]:
# Check that the file is up-to-date
# If the data exists and the last row number is smaller than the last row number on Socrata, re-download
nrow_in_data = cta_df.shape[0]
print(f'Number of rows in the data: {nrow_in_data}')

# Check against Socrata
url = "https://data.cityofchicago.org/resource/5neh-572f.json"
params = {
    "$select": "count(*)"
}

data_socrata_json = requests.get(url, params=params).json()
nrow_in_socrata = int(data_socrata_json[0]['count'])

if nrow_in_data == nrow_in_socrata:
    print('The local data is up-to-date.')
else:
    pass


Number of rows in the data: 1285295
[{'count': '1285295'}]
